# Getting started with ftmwpipeline

This notebook processes one FTMW experiment from a raw free-induction
decay (FID) to a fitted line list, then inspects the intermediate
results. It uses the bundled Blackchirp experiment `2638` and the
stateless functional API (`ftmwpipeline.api`).

It mirrors `examples/basic_usage.py` and the *Quickstart* page of the
documentation; the `Pipeline` class and the `ftmwpipeline` command line
drive the same stages.

## Setup

Confirm the package imports and locate the example experiment. The data
ships with the source repository (not the installed package), so run this
notebook from a checkout.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import ftmwpipeline.api as ftmw
from ftmwpipeline.workflows import validate_installation

validate_installation()

In [ ]:
# Find the bundled example experiment by walking up from the working
# directory, so the notebook runs from the repo root or its own folder.
def _find_example():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in ('examples/blackchirp_data/2638', 'blackchirp_data/2638'):
            cand = base / rel
            if cand.exists():
                return cand
    raise FileNotFoundError('example experiment 2638 not found')

example_data = _find_example()
example_data

## Import the FID

Each experiment becomes a single self-contained `.ftmw` HDF5 file. We
write it into a temporary directory so the notebook leaves nothing
behind; point `work` at a real path to keep the file.

In [ ]:
import tempfile

work = Path(tempfile.mkdtemp(prefix='ftmw_nb_'))
ftmw_file = work / 'exp_2638.ftmw'

fid = ftmw.import_data(ftmw_file, source=str(example_data))
fid

## Fourier transform

Experiment 2638's active spectral band is 26500-40000 MHz, so the
canonical transform is trimmed to that range. The transform itself is
unapodized and native-length.

In [ ]:
ft = ftmw.compute_ft(ftmw_file, trim=(26500, 40000))

freqs = np.linspace(ft.freq_range[0], ft.freq_range[1], ft.n_points)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(freqs, ft.magnitude_spectrum, lw=0.5)
ax.set_xlabel('frequency (MHz)')
ax.set_ylabel('magnitude')
ax.set_title('Experiment 2638 — active-band Fourier transform')
plt.show()

## Noise and decay-time calibration

Stage 2 estimates the per-bin noise that every later stage scores
against. Stage 2b extracts the data-driven decay time used to anchor the
fit and to choose the line shape.

In [ ]:
noise = ftmw.estimate_noise(ftmw_file)
tau = ftmw.calibrate_tau(ftmw_file)
noise, tau

## Peak detection and window assignment

Stage 3 locates candidate peaks; Stage 4 groups them into the windows
the fitter will solve.

In [ ]:
peaks = ftmw.detect_peaks(ftmw_file)
plan = ftmw.assign_windows(ftmw_file)
print(f'{len(peaks)} peaks detected across {plan.n_windows} windows')

## Fit

Stage 5 fits each window's lines on the active-portion FT, producing the
merged fitted-peak list.

In [ ]:
fit = ftmw.fit_peaks(ftmw_file)
print(f'{fit.n_fitted_peaks} lines fitted across {fit.n_windows} windows')

In [ ]:
# The brightest few fitted lines, by amplitude.
brightest = sorted(fit.fitted_peaks, key=lambda p: p.amplitude, reverse=True)[:8]
for p in brightest:
    print(f'{p.frequency_mhz:12.4f} MHz   amplitude {p.amplitude:.4g}')

## Next steps

* `ftmwpipeline info <file>` and `ftmwpipeline fit show <file>` inspect a
  finished file from the command line.
* The *Settings and presets* documentation covers tuning each stage's
  parameters.
* `ftmwpipeline review run` followed by `ftmwpipeline report run` produces
  the finalized line list and an HTML report.